In [2]:
from pathlib import Path
import re
import pandas as pd
from typing import Union, List, Optional

def _to_number(s: str) -> Union[int, float]:
    """Cast '256.0' → 256 (int) and '256.5' → 256.5 (float)."""
    if "." in s:
        as_float = float(s)
        return int(as_float) if as_float.is_integer() else as_float
    return int(s)

def _parse_cu_mask(raw: str, *, to_int: bool) -> Optional[Union[int, float, str]]:
    """Handle hex/dec/nan mask; return int, float('nan'), or raw string."""
    if raw.lower() == "nan":
        return float("nan") if to_int else raw
    if raw.startswith("0x"):
        return int(raw, 16) if to_int else raw
    return _to_number(raw) if to_int else raw

def load_kernel_traces(
    directory: Path | str = ".",
    *,
    as_concat: bool = True,
    cu_mask_as_int: bool = False,
) -> Union[pd.DataFrame, List[pd.DataFrame]]:
    directory = Path(directory)
    trace_files = sorted(directory.glob("*_kernel_trace.csv"))

    # Compile all three patterns with explicit modes
    patterns = [
        (
            "prefill_and_decode",
            re.compile(
                r"^standalone_ffn_tp8_"
                r"(?P<prefill_batch>\d+(?:\.\d+)?)_"
                r"(?P<prefill_len>\d+(?:\.\d+)?)_"
                r"(?P<decode_batch_size>\d+(?:\.\d+)?)_"
                # r"(?P<decode_len>\d+(?:\.\d+)?)_"
                r"(?P<cu_mask>0x[0-9A-Fa-f]+|\d+(?:\.\d+)?|nan)"
                r"_kernel_trace\.csv$"
            ),
        ),
        (
            "prefill",
            re.compile(
                r"^standalone_ffn_prefill_tp8_"
                r"(?P<prefill_batch>\d+(?:\.\d+)?)_"
                r"(?P<prefill_len>\d+(?:\.\d+)?)_"
                r"(?P<cu_mask>0x[0-9A-Fa-f]+|\d+(?:\.\d+)?|nan)"
                r"_kernel_trace\.csv$"
            ),
        ),
        (
            "decode",
            re.compile(
                r"^standalone_ffn_decode_tp8_"
                r"(?P<decode_batch_size>\d+(?:\.\d+)?)_"
                # r"(?P<decode_len>\d+(?:\.\d+)?)_"
                r"(?P<cu_mask>0x[0-9A-Fa-f]+|\d+(?:\.\d+)?|nan)"
                r"_kernel_trace\.csv$"
            ),
        ),
    ]

    def _maybe_num(v: Optional[str]):
        return float("nan") if v is None else _to_number(v)

    frames = []
    for f in trace_files:
        matched = None
        mode = None
        for m_mode, pat in patterns:
            m = pat.match(f.name)
            if m:
                matched = m
                mode = m_mode
                break

        if not matched:
            print(f"⚠️  Skipping file that doesn’t match known patterns: {f.name}")
            continue

        df = pd.read_csv(f)
        df["mode"] = mode

        # Fill all expected columns; use NaN for those absent in the filename
        df["prefill_batch"]     = _maybe_num(matched.groupdict().get("prefill_batch"))
        df["prefill_len"]       = _maybe_num(matched.groupdict().get("prefill_len"))
        df["decode_batch_size"] = _maybe_num(matched.groupdict().get("decode_batch_size"))
        df["decode_len"]        = _maybe_num(matched.groupdict().get("decode_len"))

        # cu_mask is always present in all three patterns
        df["cu_mask"] = _parse_cu_mask(matched["cu_mask"], to_int=cu_mask_as_int)
        df["source_file"] = f.name

        frames.append(df)

    if not frames:
        raise FileNotFoundError("No *_kernel_trace.csv files matched the expected patterns.")

    return pd.concat(frames, ignore_index=True) if as_concat else frames

# Example:
# standalone_attn_prefill_tp8_8_8192_32_kernel_trace.csv
# standalone_attn_decode_tp8_16_64_0xFF_kernel_trace.csv
# standalone_attn_prefill_and_decode_tp8_8_8192_16_64_nan_kernel_trace.csv
if __name__ == "__main__":
    kernel_df = load_kernel_traces(".", cu_mask_as_int=True)
    print(f"Loaded {kernel_df.source_file.nunique()} files; shape = {kernel_df.shape}")


⚠️  Skipping file that doesn’t match known patterns: standalone_ffn_16_1024_128_128_kernel_trace.csv
⚠️  Skipping file that doesn’t match known patterns: standalone_ffn_16_1024_128_160_kernel_trace.csv
⚠️  Skipping file that doesn’t match known patterns: standalone_ffn_16_1024_128_32_kernel_trace.csv
⚠️  Skipping file that doesn’t match known patterns: standalone_ffn_16_1024_128_64_kernel_trace.csv
⚠️  Skipping file that doesn’t match known patterns: standalone_ffn_16_1024_128_96_kernel_trace.csv
⚠️  Skipping file that doesn’t match known patterns: standalone_ffn_16_1024_128_nan_kernel_trace.csv
⚠️  Skipping file that doesn’t match known patterns: standalone_ffn_16_1024_16_128_kernel_trace.csv
⚠️  Skipping file that doesn’t match known patterns: standalone_ffn_16_1024_16_160_kernel_trace.csv
⚠️  Skipping file that doesn’t match known patterns: standalone_ffn_16_1024_16_32_kernel_trace.csv
⚠️  Skipping file that doesn’t match known patterns: standalone_ffn_16_1024_16_64_kernel_trace.csv

In [3]:
kernel_df['cu_mask']=kernel_df['cu_mask'].fillna(304)
mask = kernel_df["Kernel_Name"].str.startswith(
    ("Cijk"),
    na=False
)
attn_df = kernel_df[mask]

In [4]:
attn_df['mode'].value_counts()

mode
prefill_and_decode    14400
prefill                 900
decode                  240
Name: count, dtype: int64

In [5]:
## Ensuring Corerct number of rows and in correct order

import numpy as np
import pandas as pd

comb_cols = [
    "mode",
    "prefill_batch",
    "prefill_len",
    "decode_batch_size",
    "decode_len",
    "cu_mask",
]

attn_df = attn_df.copy().reset_index(drop=False).rename(columns={"index": "_orig"})

ordered_attn_df = (
    attn_df
    .assign(
        _block_order=attn_df.groupby(comb_cols, sort=False, dropna=False).ngroup()
    )
    .sort_values(["_block_order", "_orig"], kind="stable")
    .drop(columns="_block_order")
    .reset_index(drop=True)
)



print(f"Final shape: {ordered_attn_df.shape}")
# ordered_attn_df now has blocks of 5 or 10 rows in the right order


Final shape: (15540, 26)


In [6]:
ordered_attn_df["duration_us"] = (
    ordered_attn_df["End_Timestamp"] - ordered_attn_df["Start_Timestamp"]
) / 1_000

In [7]:
ordered_attn_df.to_csv("all_data_tp8.csv")

In [11]:
import pandas as pd

# Columns to group by
grp_cols = ["mode", "prefill_len", "prefill_batch", "decode_batch_size", "cu_mask"]

# Sanity check for columns
missing = [c for c in grp_cols if c not in ordered_attn_df.columns]
if missing:
    raise KeyError(f"Missing columns in ordered_attn_df: {missing}")

# Count rows per group (keep NaNs as their own groups)
counts = (
    ordered_attn_df
    .groupby(grp_cols, dropna=False)
    .size()
    .rename("count")
    .reset_index()
)

# Expected count: 10 for mode == 'prefill_and_decode', else 5
counts["expected"] = np.where(counts["mode"].eq("prefill_and_decode"), 10, 5)

# Find violations
bad = counts[counts["count"] != counts["expected"]]

if bad.empty:
    print("✅ All groups have the expected number of rows (5; or 10 when mode='prefill_and_decode').")
else:
    print("❌ These groups violate the expected row counts (showing actual vs expected):")
    print(bad.sort_values(grp_cols).to_string(index=False))
    # Optional: show the actual offending rows
    viol_rows = ordered_attn_df.merge(bad[grp_cols], on=grp_cols, how="inner")
    print("\nSample offending rows:")
    print(viol_rows.sort_values(grp_cols).head(20).to_string(index=False))
    raise AssertionError(f"{len(bad)} group(s) violate expected counts.")

✅ All groups have the expected number of rows (5; or 10 when mode='prefill_and_decode').


In [20]:
# ---------- sanity checks ----------
need = set(grp_cols + ["duration_us"])
missing = [c for c in need if c not in ordered_attn_df.columns]
if missing:
    raise KeyError(f"Missing columns in ordered_attn_df: {missing}")

# detect queue id column (needed for mode='prefill_and_decode')
queue_col = None
for cand in ["queue id", "queue_id", "Queue_Id", "Queue ID"]:
    if cand in ordered_attn_df.columns:
        queue_col = cand
        break
if queue_col is None:
    raise KeyError("Expected a queue id column (e.g., 'queue id' or 'queue_id') for PAD rows.")

# Helper to normalize queue ids to ints when possible (e.g., '1' or 1.0 -> 1)
def _norm_qid(x):
    try:
        return int(float(x))
    except Exception:
        return x

# ============================================================
# A) NON-PAD (modes != 'prefill_and_decode'): use LAST 3 of 5
# ============================================================
non_pad = ordered_attn_df[ordered_attn_df["mode"] != "prefill_and_decode"].copy()

# If needed, sort within groups before tail(3) (e.g., by a timestamp/iteration column)
# non_pad = non_pad.sort_values(grp_cols + ["iteration_or_time"])

last3 = (
    non_pad
    .groupby(grp_cols, dropna=False, group_keys=False)
    .tail(3)
)

agg_non_pad = (
    last3
    .groupby(grp_cols, dropna=False)["duration_us"]
    .agg(mean="mean", std="std")
    .reset_index()
)

# Put stats into Queue_Id=1 columns; leave 2 and 3 as NaN
non_pad_wide = agg_non_pad[grp_cols].copy()
non_pad_wide["mean_q1"] = agg_non_pad["mean"]
non_pad_wide["std_q1"]  = agg_non_pad["std"]
for q in (2, 3):
    non_pad_wide[f"mean_q{q}"] = np.nan
    non_pad_wide[f"std_q{q}"]  = np.nan

# =========================================
# B) PAD (mode == 'prefill_and_decode')
#    group by queue id, 5 rows each
# =========================================
pad = ordered_attn_df[ordered_attn_df["mode"] == "prefill_and_decode"].copy()
pad["qid_norm"] = pad[queue_col].apply(_norm_qid)

stats_pad = (
    pad
    .groupby(grp_cols + ["qid_norm"], dropna=False)["duration_us"]
    .agg(mean="mean", std="std")
    .reset_index()
)

# Pivot to wide with fixed column order: Queue_Id 1, 2, 3 (fill missing with NaN)
means = stats_pad.pivot(index=grp_cols, columns="qid_norm", values="mean")
stds  = stats_pad.pivot(index=grp_cols, columns="qid_norm", values="std")

# ensure integer columns where possible; then reindex to [1,2,3]
target_qids = [1, 2, 3]
means = means.reindex(columns=target_qids)
stds  = stds.reindex(columns=target_qids)

means.columns = [f"mean_q{q}" for q in means.columns]
stds.columns  = [f"std_q{q}"  for q in stds.columns]

pad_wide = (
    means.join(stds)
         .reindex(columns=["mean_q1","std_q1","mean_q2","std_q2","mean_q3","std_q3"])
         .reset_index()
)

# =========================================
# C) Combine NON-PAD and PAD into one table
# =========================================
combined_wide = pd.concat([non_pad_wide, pad_wide], ignore_index=True, sort=False)

# stable sort for readability
combined_wide = combined_wide.sort_values(grp_cols, kind="mergesort").reset_index(drop=True)

# Optional: enforce final column order
combined_wide = combined_wide[
    grp_cols + ["mean_q1","std_q1","mean_q2","std_q2","mean_q3","std_q3"]
]

# `combined_wide` now has, for every (mode, prefill len, prefill batch, decode batch, cu mask):
# - Queue_Id=1 mean/std in columns mean_q1/std_q1,
# - Queue_Id=2 mean/std in columns mean_q2/std_q2,
# - Queue_Id=3 mean/std in columns mean_q3/std_q3.
# For non-PAD groups, stats are computed from the last 3 rows and placed under q1; q2/q3 are NaN.


In [21]:
combined_wide

,mode,prefill_len,prefill_batch,decode_batch_size,cu_mask,mean_q1,std_q1,mean_q2,std_q2,mean_q3,std_q3
0,decode,NaN,NaN,1.0,32.0,40.292000,0.361000,NaN,NaN,NaN,NaN
1,decode,NaN,NaN,1.0,64.0,21.743000,0.180851,NaN,NaN,NaN,NaN
2,decode,NaN,NaN,1.0,96.0,20.473333,0.442945,NaN,NaN,NaN,NaN
3,decode,NaN,NaN,1.0,128.0,14.486667,0.504382,NaN,NaN,NaN,NaN
4,decode,NaN,NaN,1.0,160.0,13.845000,0.122748,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1663,prefill_and_decode,8192.0,16.0,128.0,64.0,NaN,NaN,14309.1100,510.851662,105.7620,19.450823
1664,prefill_and_decode,8192.0,16.0,128.0,96.0,NaN,NaN,15377.1216,537.052724,77.8580,12.875431
1665,prefill_and_decode,8192.0,16.0,128.0,128.0,NaN,NaN,17199.4914,417.549246,55.2384,6.733023
1666,prefill_and_decode,8192.0,16.0,128.0,160.0,NaN,NaN,19791.8150,598.074907,49.4650,6.891430


In [22]:
combined_wide.to_csv("linear_tp8_analysis.csv")

In [23]:
import numpy as np
import pandas as pd

# --- config ---
key_cols = [c for c in grp_cols if c != "mode"]

# 1) Start from PAD rows only
pad = combined_wide[combined_wide["mode"] == "prefill_and_decode"].copy()
result = pad.copy()

# 2) Attach mean_q1/std_q1 from each other mode, treating NaNs in OTHER modes' group cols as wildcards
other_modes = (
    combined_wide.loc[combined_wide["mode"] != "prefill_and_decode", "mode"]
    .dropna()
    .unique()
)

def attach_mode_stats(result_df: pd.DataFrame, src: pd.DataFrame, mode_name: str) -> pd.DataFrame:
    """Attach {mode}_mean_q1 and {mode}_std_q1 to result_df by wildcard-matching on NaNs in src group cols."""
    cols_needed = grp_cols + ["mean_q1", "std_q1"]
    if not set(cols_needed).issubset(src.columns):
        raise KeyError(f"Source for mode '{mode_name}' missing required columns: {set(cols_needed) - set(src.columns)}")

    dfm = src[cols_needed].copy()

    # drop exact duplicates; compute specificity (# of non-NaN keys); process less specific first so more specific overwrites
    dfm = dfm.drop_duplicates()
    dfm["__specificity__"] = dfm[key_cols].notna().sum(axis=1)
    dfm = dfm.sort_values(["__specificity__"])  # less specific (more NaNs) first

    out_mean = f"{mode_name}_mean_q1"
    out_std  = f"{mode_name}_std_q1"
    if out_mean not in result_df.columns:
        result_df[out_mean] = np.nan
    if out_std not in result_df.columns:
        result_df[out_std] = np.nan

    # row-wise apply: NaN in src key acts as wildcard (matches all values)
    for _, r in dfm.iterrows():
        mask = pd.Series(True, index=result_df.index)
        for c in key_cols:
            if pd.notna(r[c]):
                mask &= (result_df[c] == r[c])
                # (we only constrain on columns specified in src; NaN there => wildcard)
        # assign; later (more specific) rows overwrite earlier assignments
        result_df.loc[mask, out_mean] = r["mean_q1"]
        result_df.loc[mask, out_std]  = r["std_q1"]

    return result_df

# 3) Build a per-mode source and attach
for m in other_modes:
    src_m = combined_wide[combined_wide["mode"] == m]
    result = attach_mode_stats(result, src_m, m)

# `result` now contains only mode='prefill_and_decode' rows,
# plus extra columns like `{mode}_mean_q1` and `{mode}_std_q1`
# (with NaN wildcards from other modes treated as "match anything").


In [24]:
result

,mode,prefill_len,prefill_batch,decode_batch_size,cu_mask,mean_q1,std_q1,mean_q2,std_q2,mean_q3,std_q3,decode_mean_q1,decode_std_q1,prefill_mean_q1,prefill_std_q1
228,prefill_and_decode,256.0,1.0,1.0,32.0,NaN,NaN,47.4284,1.932905,38.2396,2.444472,40.292000,0.361000,40.880333,0.498078
229,prefill_and_decode,256.0,1.0,1.0,64.0,NaN,NaN,74.5384,4.825524,22.6840,1.985730,21.743000,0.180851,63.732667,0.061101
230,prefill_and_decode,256.0,1.0,1.0,96.0,NaN,NaN,75.1716,5.950955,23.6140,1.206453,20.473333,0.442945,62.262333,0.120001
231,prefill_and_decode,256.0,1.0,1.0,128.0,NaN,NaN,76.7438,7.560831,21.6734,2.604941,14.486667,0.504382,60.511667,0.197890
232,prefill_and_decode,256.0,1.0,1.0,160.0,NaN,NaN,75.3086,7.058589,22.4592,1.825906,13.845000,0.122748,61.206667,0.863931
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1663,prefill_and_decode,8192.0,16.0,128.0,64.0,NaN,NaN,14309.1100,510.851662,105.7620,19.450823,74.356667,0.381175,13697.847000,105.808049
1664,prefill_and_decode,8192.0,16.0,128.0,96.0,NaN,NaN,15377.1216,537.052724,77.8580,12.875431,57.598667,0.350449,14789.919000,40.653739
1665,prefill_and_decode,8192.0,16.0,128.0,128.0,NaN,NaN,17199.4914,417.549246,55.2384,6.733023,41.575333,0.312986,16747.665000,24.080938
1666,prefill_and_decode,8192.0,16.0,128.0,160.0,NaN,NaN,19791.8150,598.074907,49.4650,6.891430,40.185333,0.129516,19172.894000,60.189729


In [25]:
result.to_csv("linear_tp8_final.csv")

In [3]:
import pandas as pd
ordered_attn_df=pd.read_csv("all_data_tp8.csv")
overlap=ordered_attn_df[ordered_attn_df['mode']=='prefill_and_decode']
overlap

,Unnamed: 0,_orig,Kind,Agent_Id,Queue_Id,Thread_Id,Dispatch_Id,Kernel_Id,Kernel_Name,Correlation_Id,...,Grid_Size_Y,Grid_Size_Z,mode,prefill_batch,prefill_len,decode_batch_size,decode_len,cu_mask,source_file,duration_us
1140,1140,4804,KERNEL_DISPATCH,3,2,555731,17,3355,Cijk_Alik_Bljk_H_HS_BH_Bias_HA_S_SAV_UserArgs_...,17,...,1,1,prefill_and_decode,16.0,1024.0,128.0,NaN,128.0,standalone_ffn_tp8_16_1024_128_128_kernel_trac...,2406.105
1141,1141,4805,KERNEL_DISPATCH,3,3,555731,18,4966,Cijk_Alik_Bljk_HHS_BH_MT64x64x128_MI16x16x16x1...,18,...,2,1,prefill_and_decode,16.0,1024.0,128.0,NaN,128.0,standalone_ffn_tp8_16_1024_128_128_kernel_trac...,46.867
1142,1142,4806,KERNEL_DISPATCH,3,3,555731,20,4966,Cijk_Alik_Bljk_HHS_BH_MT64x64x128_MI16x16x16x1...,20,...,2,1,prefill_and_decode,16.0,1024.0,128.0,NaN,128.0,standalone_ffn_tp8_16_1024_128_128_kernel_trac...,63.385
1143,1143,4807,KERNEL_DISPATCH,3,2,555731,19,3355,Cijk_Alik_Bljk_H_HS_BH_Bias_HA_S_SAV_UserArgs_...,19,...,1,1,prefill_and_decode,16.0,1024.0,128.0,NaN,128.0,standalone_ffn_tp8_16_1024_128_128_kernel_trac...,2346.048
1144,1144,4808,KERNEL_DISPATCH,3,3,555731,22,4966,Cijk_Alik_Bljk_HHS_BH_MT64x64x128_MI16x16x16x1...,22,...,2,1,prefill_and_decode,16.0,1024.0,128.0,NaN,128.0,standalone_ffn_tp8_16_1024_128_128_kernel_trac...,56.650
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15535,15535,42223,KERNEL_DISPATCH,3,2,503791,21,2521,Cijk_Alik_Bljk_H_HS_BH_UserArgs_MT256x192x64_M...,21,...,1,1,prefill_and_decode,8.0,8192.0,8.0,NaN,304.0,standalone_ffn_tp8_8_8192_8_nan_kernel_trace.csv,6527.456
15536,15536,42224,KERNEL_DISPATCH,3,3,503791,24,2989,Cijk_Alik_Bljk_HHS_BH_MT32x16x256_MI16x16x16x1...,24,...,1,1,prefill_and_decode,8.0,8192.0,8.0,NaN,304.0,standalone_ffn_tp8_8_8192_8_nan_kernel_trace.csv,168.867
15537,15537,42225,KERNEL_DISPATCH,3,2,503791,23,2521,Cijk_Alik_Bljk_H_HS_BH_UserArgs_MT256x192x64_M...,23,...,1,1,prefill_and_decode,8.0,8192.0,8.0,NaN,304.0,standalone_ffn_tp8_8_8192_8_nan_kernel_trace.csv,6318.498
15538,15538,42226,KERNEL_DISPATCH,3,3,503791,26,2989,Cijk_Alik_Bljk_HHS_BH_MT32x16x256_MI16x16x16x1...,26,...,1,1,prefill_and_decode,8.0,8192.0,8.0,NaN,304.0,standalone_ffn_tp8_8_8192_8_nan_kernel_trace.csv,167.904


In [4]:
qcol = "Queue_Id"
grp_cols = ["mode", "prefill_len", "prefill_batch", "decode_batch_size", "cu_mask"]

# drop the first 2 rows within each (group cols + queue id) group (leaving rows 3,4,5)
cum = overlap.groupby(grp_cols + [qcol], dropna=False).cumcount()
overlap_trimmed = overlap.loc[cum >= 2].copy()

# --- optional: verify we now have exactly 3 rows per subgroup ---
sizes = overlap_trimmed.groupby(grp_cols + [qcol], dropna=False).size()
bad = sizes[sizes != 3]
if not bad.empty:
    print("⚠️ Some (group, Queue_Id) subgroups do not have exactly 3 rows after trimming:")
    print(bad)

In [9]:
overlap_trimmed = (
    overlap_trimmed
    .sort_values("Start_Timestamp", kind="mergesort", na_position="last")
    .reset_index(drop=True)
)

In [10]:
import numpy as np
import pandas as pd

# detect queue id / time columns
qcol = "Queue_Id"

start_col ="Start_Timestamp"
end_col   = "End_Timestamp"
# ensure datetime
for c in [start_col, end_col]:
    if not np.issubdtype(overlap_trimmed[c].dtype, np.datetime64):
        overlap_trimmed[c] = pd.to_datetime(overlap_trimmed[c], errors="coerce")

def _sum_pair_durations(g: pd.DataFrame) -> pd.Series:
    # Assumes rows alternate q=2,3,2,3,... with the FIRST row having q=2
    g = g.copy()

    # OPTIONAL: if you have a reliable ordering column, sort within group first (uncomment & customize)
    # g = g.sort_values("your_order_col_here")

    g["next_qid"] = g[qcol].shift(-1)
    g["next_end"] = g[end_col].shift(-1)

    # keep only 2→3 pairs; duration = end(3) - start(2)
    mask = (g[qcol] == 2) & (g["next_qid"] == 3)
    durations = g.loc[mask, "next_end"] - g.loc[mask, start_col]

    return pd.Series({
        "total_duration": durations.sum(),   # pandas Timedelta
        "num_pairs": int(mask.sum())
    })

pair_durations = (
    overlap_trimmed
    .groupby(grp_cols, dropna=False, group_keys=False)
    .apply(_sum_pair_durations)
    .reset_index()
)

# `pair_durations` now has one row per (mode, prefill len, prefill batch, decode batch, cu mask)
# with the total duration computed as sum over (end_ts of q=3) - (start_ts of preceding q=2).


/tmp/ipykernel_643525/2353326334.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_sum_pair_durations)


In [11]:
pair_durations['num_pairs'].value_counts()

num_pairs
3    1440
Name: count, dtype: int64

In [15]:
import numpy as np
import pandas as pd

# --- config ---
qcol = "Queue_Id"
start_col = "Start_Timestamp"
end_col   = "End_Timestamp"

df = overlap_trimmed.copy()

# 1) Ensure timestamps are timezone-aware datetimes (ns -> datetime)
for c in [start_col, end_col]:
    if np.issubdtype(df[c].dtype, np.number):
        df[c] = pd.to_datetime(df[c], unit="ns", utc=True)
    elif not np.issubdtype(df[c].dtype, np.datetime64):
        df[c] = pd.to_datetime(df[c], errors="coerce", utc=True)

# 2) Group-wise pairing: assume rows alternate 2,3,2,3 *after your trimming*.
#    We'll still sort by start time inside each group to be safe.
def _pair_stats(g: pd.DataFrame) -> pd.Series:
    g = g.sort_values(start_col).copy()

    # pair row i (qid=2) with row i+1 (qid=3)
    g["next_qid"]   = g[qcol].shift(-1)
    g["next_start"] = g[start_col].shift(-1)
    g["next_end"]   = g[end_col].shift(-1)

    mask = (g[qcol] == 2) & (g["next_qid"] == 3)

    # total span per pair: end(3) - start(2)
    span = g.loc[mask, "next_end"] - g.loc[mask, start_col]

    # overlap per pair between intervals [start2, end2] and [start3, end3]
    s2 = g.loc[mask, start_col].astype("int64")  # ns
    e2 = g.loc[mask, end_col].astype("int64")    # ns
    s3 = g.loc[mask, "next_start"].astype("int64")
    e3 = g.loc[mask, "next_end"].astype("int64")

    start_max = np.maximum(s2.values, s3.values)
    end_min   = np.minimum(e2.values, e3.values)
    overlap_ns = np.maximum(end_min - start_max, 0)  # clip negative to 0
    overlap = pd.to_timedelta(overlap_ns, unit="ns")

    return pd.Series({
        "num_pairs": int(mask.sum()),
        "total_duration_avg": span.mean(),     # Timedelta (average of spans)
        "overlap_avg": overlap.mean(),         # Timedelta (average overlap)
    })

pair_stats = (
    df.groupby(grp_cols, dropna=False, group_keys=False)
      .apply(_pair_stats)
      .reset_index()
)

# Optional numeric forms (milliseconds)
pair_stats["total_duration_us_avg"] = pair_stats["total_duration_avg"].dt.total_seconds() * 1_000_000
pair_stats["overlap_us_avg"]        = pair_stats["overlap_avg"].dt.total_seconds() * 1_000_000

pair_stats


/tmp/ipykernel_643525/286641957.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_pair_stats)


,mode,prefill_len,prefill_batch,decode_batch_size,cu_mask,num_pairs,total_duration_avg,overlap_avg,total_duration_us_avg,overlap_us_avg
0,prefill_and_decode,256.0,1.0,1.0,32.0,3,0 days 00:00:00.000077724,0 days 00:00:00.000010610,77.724,10.610
1,prefill_and_decode,256.0,1.0,1.0,64.0,3,0 days 00:00:00.000059643,0 days 00:00:00.000023814,59.643,23.814
2,prefill_and_decode,256.0,1.0,1.0,96.0,3,0 days 00:00:00.000061781,0 days 00:00:00.000024295,61.781,24.295
3,prefill_and_decode,256.0,1.0,1.0,128.0,3,0 days 00:00:00.000058802,0 days 00:00:00.000022865,58.802,22.865
4,prefill_and_decode,256.0,1.0,1.0,160.0,3,0 days 00:00:00.000059789,0 days 00:00:00.000023333,59.789,23.333
...,...,...,...,...,...,...,...,...,...,...
1435,prefill_and_decode,8192.0,16.0,128.0,64.0,3,0 days 00:00:00.000148085,0 days 00:00:00.000105775,148.085,105.775
1436,prefill_and_decode,8192.0,16.0,128.0,96.0,3,0 days 00:00:00.000119580,0 days 00:00:00.000076468,119.580,76.468
1437,prefill_and_decode,8192.0,16.0,128.0,128.0,3,0 days 00:00:00.000099377,0 days 00:00:00.000054605,99.377,54.605
1438,prefill_and_decode,8192.0,16.0,128.0,160.0,3,0 days 00:00:00.000093522,0 days 00:00:00.000050021,93.522,50.021


In [16]:
# pair_stats
result=pd.read_csv("linear_tp8_final.csv")

In [18]:
def merge_null_equal(left, right, on, suffixes=("", "_pair")):
    L = left.copy()
    R = right.copy()
    for c in on:
        L[c] = L[c].astype("object").where(~L[c].isna(), "__NA__")
        R[c] = R[c].astype("object").where(~R[c].isna(), "__NA__")
    out = L.merge(R, on=on, how="left", sort=False, validate="m:1", suffixes=suffixes)
    for c in on:
        out[c] = out[c].replace("__NA__", np.nan)
    return out

# Keep all rows from `result`, add columns from `pair_stats`
merged = merge_null_equal(result, pair_stats, grp_cols)

# If you don't need NaN==NaN semantics, a plain merge is enough:
# merged = result.merge(pair_stats_u, on=grp_cols, how="left", sort=False, validate="m:1", suffixes=("", "_pair"))

merged

,Unnamed: 0,mode,prefill_len,prefill_batch,decode_batch_size,cu_mask,mean_q1,std_q1,mean_q2,std_q2,...,std_q3,decode_mean_q1,decode_std_q1,prefill_mean_q1,prefill_std_q1,num_pairs,total_duration_avg,overlap_avg,total_duration_us_avg,overlap_us_avg
0,228,prefill_and_decode,256.0,1.0,1.0,32.0,NaN,NaN,47.4284,1.932905,...,2.444472,40.292000,0.361000,40.880333,0.498078,3,0 days 00:00:00.000077724,0 days 00:00:00.000010610,77.724,10.610
1,229,prefill_and_decode,256.0,1.0,1.0,64.0,NaN,NaN,74.5384,4.825524,...,1.985730,21.743000,0.180851,63.732667,0.061101,3,0 days 00:00:00.000059643,0 days 00:00:00.000023814,59.643,23.814
2,230,prefill_and_decode,256.0,1.0,1.0,96.0,NaN,NaN,75.1716,5.950955,...,1.206453,20.473333,0.442945,62.262333,0.120001,3,0 days 00:00:00.000061781,0 days 00:00:00.000024295,61.781,24.295
3,231,prefill_and_decode,256.0,1.0,1.0,128.0,NaN,NaN,76.7438,7.560831,...,2.604941,14.486667,0.504382,60.511667,0.197890,3,0 days 00:00:00.000058802,0 days 00:00:00.000022865,58.802,22.865
4,232,prefill_and_decode,256.0,1.0,1.0,160.0,NaN,NaN,75.3086,7.058589,...,1.825906,13.845000,0.122748,61.206667,0.863931,3,0 days 00:00:00.000059789,0 days 00:00:00.000023333,59.789,23.333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,1663,prefill_and_decode,8192.0,16.0,128.0,64.0,NaN,NaN,14309.1100,510.851662,...,19.450823,74.356667,0.381175,13697.847000,105.808049,3,0 days 00:00:00.000148085,0 days 00:00:00.000105775,148.085,105.775
1436,1664,prefill_and_decode,8192.0,16.0,128.0,96.0,NaN,NaN,15377.1216,537.052724,...,12.875431,57.598667,0.350449,14789.919000,40.653739,3,0 days 00:00:00.000119580,0 days 00:00:00.000076468,119.580,76.468
1437,1665,prefill_and_decode,8192.0,16.0,128.0,128.0,NaN,NaN,17199.4914,417.549246,...,6.733023,41.575333,0.312986,16747.665000,24.080938,3,0 days 00:00:00.000099377,0 days 00:00:00.000054605,99.377,54.605
1438,1666,prefill_and_decode,8192.0,16.0,128.0,160.0,NaN,NaN,19791.8150,598.074907,...,6.891430,40.185333,0.129516,19172.894000,60.189729,3,0 days 00:00:00.000093522,0 days 00:00:00.000050021,93.522,50.021


In [20]:
import numpy as np
import pandas as pd

# --- config ---
baseline_cu_mask = 304
grp_cols = ["mode", "prefill_len", "prefill_batch", "decode_batch_size", "cu_mask"]
base_keys = [c for c in grp_cols if c != "cu_mask"]

# The columns you want to take from cu_mask=304 and attach to others
baseline_cols = [
    "mean_q1","std_q1","mean_q2","std_q2","mean_q3","std_q3",
    "decode_mean_q1","decode_std_q1","prefill_mean_q1","prefill_std_q1",
]


# 1) Build the baseline frame from cu_mask=304, one row per (base_keys)
baseline = (
    merged.loc[merged["cu_mask"] == baseline_cu_mask, base_keys + baseline_cols]
          .copy()
)

# 2) Prefix baseline columns to avoid collisions and join back onto ALL rows (including 304)
pref = f"baseline_"
baseline_renamed = {c: pref + c for c in baseline_cols}
baseline = baseline.rename(columns=baseline_renamed)

aug = merged.merge(baseline, on=base_keys, how="left", validate="m:1")

# 3) Optional: if you only want rows with cu_mask != 304 in the final output
# aug = aug[aug["cu_mask"] != baseline_cu_mask].copy()

# 4) Reorder columns: keys, cu_mask, then baseline (cm304_*) columns, then the rest
first = base_keys + ["cu_mask"]
baseline_order = [pref + c for c in baseline_cols]
rest = [c for c in aug.columns if c not in set(first + baseline_order)]
aug = aug[first + baseline_order + rest]

# 'aug' now has, for every row, the cu_mask=304 baseline columns
# (cm304_mean_q1, cm304_std_q1, ..., cm304_prefill_std_q1) aligned by the group keys
aug


,mode,prefill_len,prefill_batch,decode_batch_size,cu_mask,baseline_mean_q1,baseline_std_q1,baseline_mean_q2,baseline_std_q2,baseline_mean_q3,...,std_q3,decode_mean_q1,decode_std_q1,prefill_mean_q1,prefill_std_q1,num_pairs,total_duration_avg,overlap_avg,total_duration_us_avg,overlap_us_avg
0,prefill_and_decode,256.0,1.0,1.0,32.0,NaN,NaN,48.3988,2.212091,22.1706,...,2.444472,40.292000,0.361000,40.880333,0.498078,3,0 days 00:00:00.000077724,0 days 00:00:00.000010610,77.724,10.610
1,prefill_and_decode,256.0,1.0,1.0,64.0,NaN,NaN,48.3988,2.212091,22.1706,...,1.985730,21.743000,0.180851,63.732667,0.061101,3,0 days 00:00:00.000059643,0 days 00:00:00.000023814,59.643,23.814
2,prefill_and_decode,256.0,1.0,1.0,96.0,NaN,NaN,48.3988,2.212091,22.1706,...,1.206453,20.473333,0.442945,62.262333,0.120001,3,0 days 00:00:00.000061781,0 days 00:00:00.000024295,61.781,24.295
3,prefill_and_decode,256.0,1.0,1.0,128.0,NaN,NaN,48.3988,2.212091,22.1706,...,2.604941,14.486667,0.504382,60.511667,0.197890,3,0 days 00:00:00.000058802,0 days 00:00:00.000022865,58.802,22.865
4,prefill_and_decode,256.0,1.0,1.0,160.0,NaN,NaN,48.3988,2.212091,22.1706,...,1.825906,13.845000,0.122748,61.206667,0.863931,3,0 days 00:00:00.000059789,0 days 00:00:00.000023333,59.789,23.333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,prefill_and_decode,8192.0,16.0,128.0,64.0,NaN,NaN,14002.0794,251.993887,250.0044,...,19.450823,74.356667,0.381175,13697.847000,105.808049,3,0 days 00:00:00.000148085,0 days 00:00:00.000105775,148.085,105.775
1436,prefill_and_decode,8192.0,16.0,128.0,96.0,NaN,NaN,14002.0794,251.993887,250.0044,...,12.875431,57.598667,0.350449,14789.919000,40.653739,3,0 days 00:00:00.000119580,0 days 00:00:00.000076468,119.580,76.468
1437,prefill_and_decode,8192.0,16.0,128.0,128.0,NaN,NaN,14002.0794,251.993887,250.0044,...,6.733023,41.575333,0.312986,16747.665000,24.080938,3,0 days 00:00:00.000099377,0 days 00:00:00.000054605,99.377,54.605
1438,prefill_and_decode,8192.0,16.0,128.0,160.0,NaN,NaN,14002.0794,251.993887,250.0044,...,6.891430,40.185333,0.129516,19172.894000,60.189729,3,0 days 00:00:00.000093522,0 days 00:00:00.000050021,93.522,50.021


In [21]:
aug.to_csv("linear_tp8_final_summary.csv")